## 💻 SFT for Code Suggestion
*Experiment: Baseline Fine-tuning*

---

### 📌 Key Parameters
*   **Rank:** `8`
*   **Alpha:** `16`
*   **Target Gates:** `q` and `v`
*   **Epochs:** `2`
*   **Learning Rate:** `2e-4`

## 1) Import Libraries

In [ ]:
# Check GPU
!nvidia-smi


Tue Sep  8 14:00:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.0/81.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

In [2]:
import unsloth
import os
from pathlib import Path

import torch
from unsloth import FastLanguageModel
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, DatasetDict
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 2) Config (model + data + hyperparams)


In [3]:
set_seed(42)

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_URL = "data/processed/magicoder_instruct_dataset.jsonl"
SYSTEM_PROMPT = """
You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

Strict Output Rules:
1. Output ONLY the raw Python code.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop.
"""


# ====== TRAINING ======
MAX_SEQ_LENGTH = 2048
RANK = 8
ALPHA_RANK = 16
GATES = ["q_proj", "v_proj"]
NUM_EPOCHS = 2
LR = 2e-4
WARMUP_STEPS = 100

# Batch
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 8

# Outputs
WORKDIR = Path.cwd().resolve()
OUTPUT_DIR = WORKDIR / "artifacts" / "qwen2.5-coder-7B-base-lora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Memory
LOAD_IN_4BIT = False

# dtype
DTYPE = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("BASE_MODEL:", BASE_MODEL)
print("DTYPE:", DTYPE, "| LOAD_IN_4BIT:", LOAD_IN_4BIT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


BASE_MODEL: Qwen/Qwen2.5-Coder-7B-Instruct
DTYPE: bfloat16 | LOAD_IN_4BIT: False
GPU: NVIDIA A100-SXM4-40GB


## 3) Load model + Turn on LoRA (Unsloth)


In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = RANK,
    target_modules = GATES,
    lora_alpha = ALPHA_RANK,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
model.config.use_cache = False
model.print_trainable_parameters()

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print("Loaded model + LoRA OK")


==((====))==  Unsloth 2026.9.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.9.6 patched 28 layers with 28 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331
Loaded model + LoRA OK


## 4) Load dataset JSONL & Format by chat_template


In [5]:
raw_dataset = load_dataset("json", data_files=DATASET_URL)

train_test = raw_dataset["train"].train_test_split(test_size=0.2)
val_test = train_test["test"].train_test_split(test_size=0.5)

dataset = DatasetDict({
    "train": train_test["train"],
    "val": val_test["train"],
    "test": val_test["test"]
})

print(dataset)

sample = dataset["train"][0]
print("\nUser Prompt preview:\n")
print(sample["messages"][0]["content"][:500])
print("\nAssistant Code preview:\n")
print(sample["messages"][1]["content"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1600
    })
    val: Dataset({
        features: ['messages'],
        num_rows: 200
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 200
    })
})

User Prompt preview:

Create an API endpoint using FastAPI that retrieves statistics related to reviews in a review pipeline. This endpoint should be accessible at the path '/reviews/stats' and should require admin privileges for access. The function should accept a current user as a dependency, ensuring only users with admin rights can access the data.

The statistics to be computed include:
1. The total number of reviews that are currently pending across different organizations.
2. A breakdown of pending reviews by

Assistant Code preview:

@router.get('/reviews/stats', response_model=ReviewStatsResponse)
async def review_stats(current_user: User=Depends(get_current_admin), db: AsyncSession=Depends(get_session)) -> ReviewStats

In [6]:
def formatting_prompts_func(examples):
    batch_messages = examples["messages"]
    texts = []

    for messages in batch_messages:
        chat_template_messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ]

        chat_template_messages.extend(messages)

        text = tokenizer.apply_chat_template(
            chat_template_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=2)

Map (num_proc=2):   0%|          | 0/1600 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
print("Dataset size:", len(dataset["train"]))
print("\n===== PREVIEW 2 SAMPLES =====")
for i in range(2):
    print(f"\n--- Sample {i+1} ---\n{dataset['train'][i]['text']}\n...")

Dataset size: 1600

===== PREVIEW 2 SAMPLES =====

--- Sample 1 ---
<|im_start|>system

You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

Strict Output Rules:
1. Output ONLY the raw Python code.
2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop.
<|im_end|>
<|im_start|>user
Create an API endpoint using FastAPI that retrieves statistics related to reviews in a review pipeline. This endpoint should be accessible at 

## 5) Train SFT


In [9]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["val"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,

    args = SFTConfig(
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate = LR,
        num_train_epochs = NUM_EPOCHS,
        warmup_steps = WARMUP_STEPS,

        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",

        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 2,

        fp16 = (DTYPE == "float16"),
        bf16 = (DTYPE == "bfloat16"),
        optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        output_dir = OUTPUT_DIR,

        completion_only_loss = True,
        gradient_checkpointing=True
    ),
)

train_result = trainer.train()
train_result.metrics

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1600 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,600 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 2,523,136 of 7,618,139,648 (0.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:2825: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packa

Step,Training Loss,Validation Loss
20,1.788569,1.743217
40,1.683250,1.514898
60,1.368597,1.111475
80,0.974555,0.851702
100,0.848613,0.790375


Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-base-lora/checkpoint-100/tokenizer_config.json.


{'train_runtime': 435.575,
 'train_samples_per_second': 7.347,
 'train_steps_per_second': 0.23,
 'total_flos': 7.783209366172262e+16,
 'train_loss': 1.3327170372009278,
 'epoch': 2.0}

In [10]:
for callback in trainer.callback_handler.callbacks.copy():
    if "Notebook" in callback.__class__.__name__:
        trainer.remove_callback(callback)

eval_results = trainer.evaluate()
print("===== VAL SET METRICS =====")
eval_results

===== VAL SET METRICS =====


{'eval_loss': 0.7903750538825989,
 'eval_runtime': 8.8322,
 'eval_samples_per_second': 22.644,
 'eval_steps_per_second': 5.661,
 'epoch': 2.0}

In [11]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_test_dataset = dataset["test"].map(
    tokenize_function,
    batched=True,
    num_proc=2
)

test_results = trainer.evaluate(
    eval_dataset=tokenized_test_dataset,
    metric_key_prefix="test"
)

print("===== TEST SET METRICS =====")
print(test_results)

Map (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

===== TEST SET METRICS =====
{'test_loss': 0.79676353931427, 'test_runtime': 8.4765, 'test_samples_per_second': 23.595, 'test_steps_per_second': 5.899, 'epoch': 2.0}


## 6) Save LoRA Adapter

In [12]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to:", OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /content/artifacts/qwen2.5-coder-7B-base-lora/tokenizer_config.json.


Saved LoRA adapter to: /content/artifacts/qwen2.5-coder-7B-base-lora


## 7) Sanity check


In [13]:
model.to(DEVICE)
FastLanguageModel.for_inference(model)

user_question = "Write a FastAPI endpoint (async def) that accepts a long piece of text, uses Hugging Face AutoTokenizer to count the number of tokens, and returns the result. The system needs to handle thousands of concurrent requests."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_question}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(DEVICE)

outputs = model.generate(
    **inputs,
    max_new_tokens=3000,
    use_cache=True,
    temperature=0.1,
)

prompt_length = inputs["input_ids"].shape[1]
response_tokens = outputs[0][prompt_length:]
response = tokenizer.decode(response_tokens, skip_special_tokens=True)

print("===== TEST INFERENCE =====")
print(f"User: {user_question}")
print(f"Code: {response.strip()}")

Both `max_new_tokens` (=3000) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== TEST INFERENCE =====
User: Write a FastAPI endpoint (async def) that accepts a long piece of text, uses Hugging Face AutoTokenizer to count the number of tokens, and returns the result. The system needs to handle thousands of concurrent requests.
Code: @router.post("/count-tokens", response_model=TokenCountResponse)
async def count_tokens(text: str = Form(...)) -> TokenCountResponse:
    """Counts the number of tokens in the provided text."""
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenized_text = tokenizer.tokenize(text)
    return TokenCountResponse(token_count=len(tokenized_text))
